In [4]:
try:
    import os,re
    import csv
    import pandas as pd
    import numpy as np
    import warnings
    import seaborn as sns
    import matplotlib.pyplot as plt
    import fastf1
    import fastf1.plotting
    warnings.filterwarnings("ignore")
    from pathlib import Path
    from sklearn.linear_model import LogisticRegression
    from sklearn.feature_selection import RFE
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, label_binarize, MinMaxScaler
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB, MultinomialNB
    from sklearn.feature_selection import RFECV
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, auc, classification_report
    from sklearn import metrics
    import statsmodels.api as sm
    from typing import List
    import math
    import shutil
    from sklearn.pipeline import Pipeline
    print('Imported all packages successfully!')
except Exception as e:
    print(f"Exception importing the given list of packakges: {e}")
    print("Installing the following packages....")
    %pip install -r "requirements.txt"
    warnings.filterwarnings("ignore")
    print("Installation Successfull")

Imported all packages successfully!


<h4>
<b>
Let's Build a Machine Learning Model that Estimates the probability of each driver.
<hr/>
finishing on the podium in a Formula 1 Grand Prix. (Round 1 Austrailia 🐨) <br/>
<hr/>
🎯 Target Prediction -> Podium
<hr/>
Phase I: Before Saturday Qualifying 
</b>
</h4>

<h4><b>✅ Allowed Pre-Quali Features !!!</b></h4>
<div>
    <h4>1. Driver Form:</h4>
    <ul>
        <li><b>Last N race finishes</b></li>
        <li><b>Podium count (recent)</b></li>
        <li><b>Average finishing position</b></li>
        <li><b>Points in last N races</b></li>
        <li><b>DNF rate</b></li>
        <li><b>Teammate comparison</b><br/>
        <ul>
            <li><b>teammate_points</b></li>
            <li><b>teammate_podium</b></li>
            <li><b>teammate_finish_delta</b></li>
            <li><b>teammate_points_delta</b></li>
            <li><b>avg_teammate_finish_delta_last_n</b></li>
            <li><b>avg_teammate_points_delta_last_n</b></li>
        </ul>
        </li>
    </ul>
</div>
<hr/>
<div>
    <h4>2. Team Performance:</h4>
    <ul>
        <li><b>Constructor standings</b></li>
        <li><b>Average team finish</b></li>
        <li><b>Team reliability</b></li>
        <li><b>Pace ranking (season-to-date proxy)</b></li>
        <li><b>Historical team performance</b></li>
    </ul>
</div>
<hr/>
<div>
    <h4>3. Track / Circuit Context:</h4>
    <ul>
        <li><b>Circuit type (street / permanent)</b></li>
        <li><b>Downforce requirement</b></li>
        <li><b>Historical driver performance at this track</b></li>
        <li><b>Historical team performance at this track</b></li>
    </ul>
</div>
<hr/>
<div>
    <h4>4. Practice Sessions (FP1, FP2, FP3):</h4>
    <ul>
        <li><b>Best lap time</b></li>
        <li><b>Average lap time</b></li>
        <li><b>Long-run pace proxy</b></li>
        <li><b>Stint consistency</b></li>
        <li><b>Session ranking</b></li>
        <li><b>Improvement across sessions</b></li>
    </ul>
</div>
<hr/>
<div>
    <h4>5. Weekend Context:</h4>
    <ul>
        <li><b>Weather forecast</b></li>
        <li><b>Track temperature</b></li>
        <li><b>Sprint weekend indicator</b></li>
        <li><b>Penalties known before quali</b></li>
    </ul>
</div>
<hr/>

In [34]:
# Lets load and build the Driver Form Features
def load_driver_form(results_df: pd.DataFrame, n_races: int=10) -> pd.DataFrame:
    """
    Step1: Build Driver Features from historical Race Data.
    We will be using these column names throughout
    Columns:
        - Season
        - Round
        - Driver/ DriverCode
        - Constructor
        - Position
        - Points
        - Status

    Returns:
        DataFrame with one row per driver per race and lag/rolling form features.
    """
    try:
        df = results_df.copy()
        # We will be using the above column names, so we need map the ones from fastf1 to these
        rename = {}
        if "Season" not in df.columns and "year" in df.columns:
            rename["year"] = "Season"
        if "Round" not in df.columns and "round" in df.columns:
            rename["round"] = "Round"
        if "Driver" not in df.columns and "Abbreviation" in df.columns:
            rename["Abbreviation"] = "Driver"
        if "Driver" not in df.columns and "DriverCode" in df.columns:
            rename["DriverCode"] = "Driver"
        if "Driver" not in df.columns and "FullName" in df.columns:
            rename["FullName"] = "Driver"
        if "Team" not in df.columns and "TeamName" in df.columns:
            rename["TeamName"] = "Team"
        if "Team" not in df.columns and "Constructor" in df.columns:
            rename["Constructor"] = "Team"
        if "Status" not in df.columns and "ResultStatus" in df.columns:
            rename["ResultStatus"] = "Status"

        df = df.rename(columns=rename)
        required_cols = ["Season", "Round", "Driver", "Team", "Position", "Points"]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        # Clean the Columns
        df["Position"] = pd.to_numeric(df["Position"], errors="coerce")
        df["Points"] = pd.to_numeric(df["Points"], errors="coerce").fillna(0)
        
        # Retirements/Non-Finisheres/Non-Participants (especially for checo and bottas lol)
        if "Status" in df.columns:
            df["DNF"] = (~df["Status"].astype(str).str.contains("Finished", case=False)).astype(int)
        else:
            df["DNF"] = df["Position"].isna().astype(int)

        # Podium Flag for the historical Race Result --- This will one of the strongest features later as we see
        df["Podium"] = (df["Position"] <= 3).astype(int)
        
        # Sort in time order
        df = df.sort_values(["Driver", "Season", "Round"]).reset_index(drop=True)
        print("Loaded Driver's Past Form")
        
        # Create last N-races Features, group them and average their results in the df
        group = df.groupby("Driver", group_keys=False)
        for i in range(n_races + 1):
            df[f"finish_lag_{i}"] = group["Position"].shift(i)
            df[f"points_lag_{i}"] = group["Points"].shift(i)
            df[f"podium_lag_{i}"] = group["Podium"].shift(i)
            df[f"dnf_lag_{i}"] = group["DNF"].shift(i)

        df["avg_finish_in_last_n_races"] = (
            group["Position"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

        df["avg_points_in_last_n_races"] = (
            group["Points"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

        df["podium_count_in_last_n_races"] = (
            group["Podium"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).sum())
            .reset_index(level=0, drop=True)
        )

        df["dnf_rate_in_last_n_races"] = (
            group["Podium"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )
        print("Loaded Driver's Last N-Races Performance")

        # Teammate comparison Features [Head to Head]
        df["teammate_finish"] = np.nan
        df["teammate_points"] = np.nan
        df["teammate_podium"] = np.nan

        race_team_groups = df.groupby(["Season", "Round", "Team"], dropna=False)
        for (_, _, _), idx in race_team_groups.groups.items():
            race_team_rows = df.loc[idx]
            if len(race_team_rows) < 2:
                continue

            positions = race_team_rows["Position"].values
            points = race_team_rows["Points"].values
            podiums = race_team_rows["Podium"].values

            # For each driver, teammate is the other row in the same team
            for pos_idx, row_idx in enumerate(race_team_rows.index):
                teammate_pos = positions[1 - pos_idx] if len(positions) >= 2 else np.nan
                teammate_pts = points[1 - pos_idx] if len(points) >= 2 else np.nan
                teammate_pod = podiums[1 - pos_idx] if len(podiums) >= 2 else np.nan

                df.loc[row_idx, "teammate_finish"] = teammate_pos
                df.loc[row_idx, "teammate_points"] = teammate_pts
                df.loc[row_idx, "teammate_podium"] = teammate_pod

        df["teammate_finish_delta"] = df["teammate_finish"] - df["Position"]
        df["teammate_points_delta"] = df["Points"] - df["teammate_points"]

        # Rolling teammate comparison based on past races only
        df["avg_teammate_finish_delta_last_n"] = (
            group["teammate_finish_delta"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

        df["avg_teammate_points_delta_last_n"] = (
            group["teammate_points_delta"]
            .apply(lambda s: s.shift(1).rolling(n_races, min_periods=1).mean())
            .reset_index(level=0, drop=True)
        )

        print("Loaded Teammate's Head to Head Comparison")

        # Final Cleanup
        feature_cols = [
            "Season", "Round", "Driver", "Team", "Position", "Points", "Status" if "Status" in df.columns else None,
            "Podium", "DNF",
            *[f"finish_lag_{i}" for i in range(1, n_races + 1)],
            *[f"points_lag_{i}" for i in range(1, n_races + 1)],
            *[f"podium_lag_{i}" for i in range(1, n_races + 1)],
            *[f"dnf_lag_{i}" for i in range(1, n_races + 1)],
            "avg_finish_last_n",
            "podium_count_last_n",
            "avg_points_last_n",
            "dnf_rate_last_n",
            "teammate_finish",
            "teammate_points",
            "teammate_podium",
            "teammate_finish_delta",
            "teammate_points_delta",
            "avg_teammate_finish_delta_last_n",
            "avg_teammate_points_delta_last_n",
        ]
        feature_cols = [c for c in feature_cols if c is not None and c in df.columns]
        print("Driver Form features Ready!")
        return df[feature_cols].copy()
    except Exception as e:
        print(f"Error Loading Driver Form Features: \n{e}")
        return pd.DataFrame

In [28]:
"""
Lets build a helper function to load previous race 
!!!! IMP: Always cache fastf1, when using it for large data loading ops !!!
"""
CACHE_DIR = '../data/cache'
if os.path.isdir(CACHE_DIR):
    print("CACHE DIR Already Exists")
else:
    os.mkdir(CACHE_DIR)
    
fastf1.Cache.enable_cache(CACHE_DIR)
def help_load_race_data(seasons: list) -> pd.DataFrame:
    all_res = []
    for y in seasons:
        schedule = fastf1.get_event_schedule(y)
        
        for _, event in schedule.iterrows():
            try:
                session = fastf1.get_session(y, event["RoundNumber"], "R")
                session.load()

                results = session.results.copy()

                results["Season"] = y
                results["Round"] = event["RoundNumber"]
                results["EventName"] = event["EventName"]

                results = results.rename(columns={
                    "Abbreviation": "Driver",
                    "TeamName": "Team",
                    "Position": "Position",
                    "Points": "Points",
                    "Status": "Status"
                })

                results = results[[
                    "Season",
                    "Round",
                    "EventName",
                    "Driver",
                    "Team",
                    "Position",
                    "Points",
                    "Status"
                ]]

                all_res.append(results)

                print(f"Loaded {y} Round {event['RoundNumber']}")
            except Exception as e:
                print(f"Skipping {y} Round {event['RoundNumber']} because: {e}")
    return pd.concat(all_res, ignore_index=True)

CACHE DIR Already Exists


In [ ]:
# Let's Build the base Dataset
seasons = [2024, 2025]
results_df = help_load_race_data(seasons)
results_df.head(3)

In [40]:
driver_form_df = load_driver_form(results_df, n_races=15)
driver_form_df.head(5)

Loaded Driver's Past Form
Loaded Driver's Last N-Races Performance
Loaded Teammate's Head to Head Comparison
Driver Form features Ready!


,Season,Round,Driver,Team,Position,Points,Status,Podium,DNF,finish_lag_1,...,dnf_lag_13,dnf_lag_14,dnf_lag_15,teammate_finish,teammate_points,teammate_podium,teammate_finish_delta,teammate_points_delta,avg_teammate_finish_delta_last_n,avg_teammate_points_delta_last_n
0,2024,1,ALB,Williams,15.0,0.0,Lapped,0,1,NaN,...,NaN,NaN,NaN,20.0,0.0,0.0,5.0,0.0,NaN,NaN
1,2024,2,ALB,Williams,11.0,0.0,Finished,0,0,15.0,...,NaN,NaN,NaN,14.0,0.0,0.0,3.0,0.0,5.000000,0.0
2,2024,3,ALB,Williams,11.0,0.0,Lapped,0,1,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,0.0
3,2024,4,ALB,Williams,20.0,0.0,Retired,0,1,11.0,...,NaN,NaN,NaN,17.0,0.0,0.0,-3.0,0.0,4.000000,0.0
4,2024,5,ALB,Williams,12.0,0.0,Finished,0,0,20.0,...,NaN,NaN,NaN,17.0,0.0,0.0,5.0,0.0,1.666667,0.0


In [56]:
extract_cols = driver_form_df.columns[:9].tolist() + driver_form_df.columns[70:].tolist()
extract_cols

['Season',
 'Round',
 'Driver',
 'Team',
 'Position',
 'Points',
 'Status',
 'Podium',
 'DNF',
 'teammate_points',
 'teammate_podium',
 'teammate_finish_delta',
 'teammate_points_delta',
 'avg_teammate_finish_delta_last_n',
 'avg_teammate_points_delta_last_n']

In [57]:
driver_form_df_extract = driver_form_df[extract_cols].copy()
driver_form_df_extract.shape

(658, 15)

In [74]:
def load_team_data(seasons: list, n_races: int=15) -> pd.DataFrame:
    """
    Get the following Team Performance Data from the last seasons

    constructor standings
    
    average team finish
    
    team reliability
    
    pace ranking (season-to-date proxy)
    
    historical team performance
    """
    res = []
    for y in seasons:
        schedule = fastf1.get_event_schedule(y)
        for _, event in schedule.iterrows():
            round_no = int(event["RoundNumber"])
            try:
                session = fastf1.get_session(y, event["RoundNumber"], "R")
                session.load()
                
                results = session.results.copy()

                if "TeamName" in results.columns:
                    results["Team"] = results["TeamName"]
                elif "ConstructorName" in results.columns:
                    results["Team"] = results["ConstructorName"]
                else:
                    raise ValueError("No TeamName/ConstructorName column found in FastF1 results.")

                results["Season"] = y
                results["Round"] = round_no
                results["EventName"] = event["EventName"]

                results["Position"] = pd.to_numeric(results["Position"], errors="coerce")
                results["Points"] = pd.to_numeric(results["Points"], errors="coerce").fillna(0)

                finished_pattern = r"Finished|Lapped|\+\d+\sLap[s]?|Classified"
                results["Finished"] = (
                    results["Status"].astype(str).str.contains(
                        finished_pattern,
                        case=False,
                        na=False,
                        regex=True
                    )
                ).astype(int)

                results["DNF"] = (1 - results["Finished"]).astype(int)
                results["Podium"] = (results["Position"] <= 3).astype(int)

                team_race = results.groupby(
                    ["Season", "Round", "EventName", "Team"],
                    as_index=False
                ).agg(
                    team_points=("Points", "sum"),
                    avg_team_finish=("Position", "mean"),
                    best_team_finish=("Position", "min"),
                    team_podiums=("Podium", "sum"),
                    team_dnfs=("DNF", "sum"),
                    team_finishers=("Finished", "sum"),
                    drivers_count=("Team", "size")
                )

                res.append(team_race)
                print(f"Loaded {y} R{round_no}: {event['EventName']}")

            except Exception as e:
                print(f"Skipping {y} R{round_no}: {e}")

    if not res:
        return pd.DataFrame()

    team_df = pd.concat(res, ignore_index=True)
    team_df = team_df.sort_values(["Season", "Team", "Round"]).reset_index(drop=True)

    g = team_df.groupby(["Season", "Team"], group_keys=False)

    team_df["constructor_points_prev"] = g["team_points"].apply(
        lambda s: s.shift(1).cumsum()
    ).reset_index(level=0, drop=True)

    team_df["avg_finish_to_date_prev"] = g["avg_team_finish"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df["avg_points_to_date_prev"] = g["team_points"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df["dnf_rate_to_date_prev"] = g["team_dnfs"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df["reliability_to_date_prev"] = 1 - team_df["dnf_rate_to_date_prev"]

    team_df[f"avg_team_finish_last_{n_races}"] = g["avg_team_finish"].apply(
        lambda s: s.shift(1).rolling(n_races, min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df[f"team_points_last_{n_races}"] = g["team_points"].apply(
        lambda s: s.shift(1).rolling(n_races, min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df[f"team_dnf_rate_last_{n_races}"] = g["team_dnfs"].apply(
        lambda s: s.shift(1).rolling(n_races, min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    # Historical team performance at this event name
    eg = team_df.groupby(["Team", "EventName"], group_keys=False)

    team_df["historical_event_avg_finish_prev"] = eg["avg_team_finish"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df["historical_event_avg_points_prev"] = eg["team_points"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    team_df["historical_event_podium_rate_prev"] = eg["team_podiums"].apply(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    ).reset_index(level=0, drop=True)

    # Constructor standings proxy and pace rank for each race
    team_df["constructor_points_prev"] = team_df["constructor_points_prev"].fillna(0)

    team_df["constructor_standing_prev"] = team_df.groupby(["Season", "Round"])["constructor_points_prev"].rank(
        ascending=False,
        method="min"
    )

    team_df["pace_rank_prev"] = team_df.groupby(["Season", "Round"])["avg_finish_to_date_prev"].rank(
        ascending=True,
        method="min"
    )

    return team_df
                

In [75]:
"""Rememeber to cache this execution in future"""
team_performance_df = load_team_data([2024, 2025], n_races=15)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Skipping 2024 R0: Cannot get testing event by round number!


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Loaded 2024 R1: Bahrain Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

Loaded 2025 R22: Las Vegas Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '55', '4', '12', '63', '14', '16', '30', '22', '23', '44', '5', '43', '31', '10', '18', '6', '87', '27']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Loaded 2025 R23: Qatar Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '4', '16', '63', '14', '31', '44', '27', '18', '5', '87', '55', '22', '12', '23', '6', '30', '10', '43']


Loaded 2025 R24: Abu Dhabi Grand Prix


In [76]:
team_performance_df.head(3)

,Season,Round,EventName,Team,team_points,avg_team_finish,best_team_finish,team_podiums,team_dnfs,team_finishers,...,dnf_rate_to_date_prev,reliability_to_date_prev,avg_team_finish_last_15,team_points_last_15,team_dnf_rate_last_15,historical_event_avg_finish_prev,historical_event_avg_points_prev,historical_event_podium_rate_prev,constructor_standing_prev,pace_rank_prev
0,2024,1,Bahrain Grand Prix,Alpine,0.0,17.5,17.0,0,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
1,2024,2,Saudi Arabian Grand Prix,Alpine,0.0,16.5,13.0,0,1,1,...,0.0,1.0,17.5,0.0,0.0,NaN,NaN,NaN,6.0,9.0
2,2024,3,Australian Grand Prix,Alpine,0.0,14.5,13.0,0,0,2,...,0.5,0.5,17.0,0.0,0.5,NaN,NaN,NaN,7.0,10.0


In [78]:
team_performance_df.shape

(480, 24)

In [81]:
driver_team_df_merge = driver_form_df_extract.merge(
    team_performance_df,
    on=["Season", "Round", "Team"],
    how="left"
)

driver_team_df_merge.head(3)

,Season,Round,Driver,Team,Position,Points,Status,Podium,DNF,teammate_points,...,dnf_rate_to_date_prev,reliability_to_date_prev,avg_team_finish_last_15,team_points_last_15,team_dnf_rate_last_15,historical_event_avg_finish_prev,historical_event_avg_points_prev,historical_event_podium_rate_prev,constructor_standing_prev,pace_rank_prev
0,2024,1,ALB,Williams,15.0,0.0,Lapped,0,1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
1,2024,2,ALB,Williams,11.0,0.0,Finished,0,0,0.0,...,0.0,1.0,17.5,0.0,0.0,NaN,NaN,NaN,6.0,9.0
2,2024,3,ALB,Williams,11.0,0.0,Lapped,0,1,NaN,...,0.0,1.0,15.0,0.0,0.0,NaN,NaN,NaN,7.0,8.0


In [82]:
driver_team_df_merge.to_csv

(658, 36)

In [13]:
pre_quali_sessions={}
try:
    print("Loading Sessions")
    pre_quali_sessions[aus_fp1] = fastf1.get_session(2026, 1, 'FP1')
    pre_quali_sessions[aus_fp2] = fastf1.get_session(2026, 1, 'FP2')
    pre_quali_sessions[aus_fp3] = fastf1.get_session(2026, 1, 'FP3')
    print("Loded Sessions Successfully!")
except Exception as e:
    print(f"Error Loading  Session Data \n: {e}")

Loading Sessions
Loded Sessions Successfully!


In [14]:
def load_sess_data(session):
    try:
        print("Loading session data")
        session.load(telemetry=True, weather=True, messages=True)
        print(f"\n{'*'*70}")
        print(f"Event  : {session.event['EventName']}")
        print(f"Circuit: {session.event['Location']}")
        print(f"Date   : {session.event['EventDate'].strftime('%B %d, %Y')}")
        print(f"Type   : {session.name}")
        print(f"{'*'*70}")
    except Exception as e:
        print(f"Error Loading Session Details: {e}")

In [16]:
for k, v in pre_quali_sessions.items():
    load_sess_data(pre_quali_sessions[k])

core           INFO 	Loading data for Australian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Loading session data


core        WARNING 	No lap data for driver 14
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 14)
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Australian Grand Prix - Practice 2 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...



**********************************************************************
Event  : Australian Grand Prix
Circuit: Melbourne
Date   : March 08, 2026
Type   : Practice 1
**********************************************************************
Loading session data


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to


**********************************************************************
Event  : Australian Grand Prix
Circuit: Melbourne
Date   : March 08, 2026
Type   : Practice 2
**********************************************************************
Loading session data


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to


**********************************************************************
Event  : Australian Grand Prix
Circuit: Melbourne
Date   : March 08, 2026
Type   : Practice 3
**********************************************************************


In [17]:
pre_quali_sessions[aus_fp1]

2026 Season Round 1: Australian Grand Prix - Practice 1